# Data Reading

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [ ]:
df = spark.read.format("delta")\
    .load("s3://learn-databricks-project-e2e-1-bronze/orders")
display(df)

In [ ]:
df.printSchema()

In [ ]:
df = df.withColumnRenamed("_rescued_data", "rescued_data")

In [ ]:
df = df.drop("rescued_data")
display(df)

In [ ]:
df = df.withColumn("order_date", to_timestamp(col("order_date")))
df.show()

In [ ]:
df = df.withColumn("year", year(col("order_date")))\
    .withColumn("month", month(col("order_date")))\
    .withColumn("day", dayofmonth(col("order_date")))
    
df.show()

In [ ]:
df1 = df.withColumn("flag",dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
df1.show()

In [ ]:
df2 = df1.withColumn("rank_flag", rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
df2.show()

In [ ]:
df3 = df2.withColumn("row_flag", row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
df3.show()

# Classes - OOP

In [ ]:
class windows:
       
    def dense_rank(self, df):
        df_dense_rank = df.withColumn("dense_rank_flag", dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_dense_rank
    
    def rank(self, df):
        df_rank = df.withColumn("rank_flag", rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_rank
    
    def row_number(self, df):
        df_row_number = df.withColumn("row_flag", row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_row_number        

In [ ]:
obj = windows()

In [ ]:
df_result = obj.dense_rank(df)
df_result.show()

# Data Writing

In [ ]:
df.write.format("delta")\
    .mode("append")\
    .save("s3://learn-databricks-project-e2e-1-silver/orders")